# Configurations

In [ ]:
import os

from netbalance.configs.common import MODEL_SAVED_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR
from netbalance.configs.general import ModelConfig, OptimizerConfig

BLINSYN_RAW_DATA_DIR = os.path.join(RAW_DATA_DIR, "blinsyn")
BLINSYN_PROCESSED_DATA_DIR = os.path.join(PROCESSED_DATA_DIR, "blinsyn")
BLINSYN_MODEL_SAVED_DIR = os.path.join(MODEL_SAVED_DIR, "blinsyn")
BLINSYN_RESULTS_DIR = os.path.join(BLINSYN_PROCESSED_DATA_DIR, "results")

BLINSYN_CELL_FEATURES_DIR = os.path.join(BLINSYN_PROCESSED_DATA_DIR, "features/cells")
BLINSYN_DRUG_FEATURES_DIR = os.path.join(BLINSYN_PROCESSED_DATA_DIR, "features/drugs")


class BLINSYNModelConfig(ModelConfig):
    def __init__(self):
        super().__init__()

        self.drug_feature_name = "C4"
        self.cell_feature_name = "Cell2"

    def get_configuration(self):
        return super().get_configuration()

    def get_feature_extractor_kwargs(self):
        return {}

    def get_model_kwargs(self):
        return {}


class BLINSYNOptimizerConfig(OptimizerConfig):

    def __init__(self) -> None:
        super().__init__()
        self.exp_name = "BLINSYN optimizer"
        self.threshold = 0.5

    def get_configuration(self):
        return super().get_configuration()

    def get_fe_loader_configuration(self):
        return {}

# Model

In [ ]:
from pathlib import Path
from typing import Union

import numpy as np
from sklearn.linear_model import LogisticRegression

from netbalance.methods.general import FeatureExtractor

from netbalance.models.interface import AModelHandler, HandlerFactory


class BLINSYNFeatureExtractor(FeatureExtractor):

    def __init__(
        self,
        drug_feature_name,
        cell_feature_name,
    ):
        super().__init__()

        self.drug_feature_name = drug_feature_name
        self.cell_feature_name = cell_feature_name
        self.drug_file = Path(f"{BLINSYN_DRUG_FEATURES_DIR}/{drug_feature_name}.txt")
        self.cell_file = Path(f"{BLINSYN_CELL_FEATURES_DIR}/{cell_feature_name}.txt")

    def build(self):
        self.drug_features = np.loadtxt(self.drug_file, delimiter=",")
        self.cell_features = np.loadtxt(self.cell_file, delimiter=",")

    def extract_features(
        self,
        a_nodes: Union[list[int], np.ndarray],
        b_nodes: Union[list[int], np.ndarray],
        c_nodes: Union[list[int], np.ndarray],
    ):
        d1 = self.drug_features[a_nodes]
        d2 = self.drug_features[b_nodes]
        c = self.cell_features[c_nodes]
        features = np.concatenate([d1, d2, c], axis=1)
        return features


class BLINSYNModelHandler(AModelHandler):
    """Baseline model which generates a random number between 0 and 1 as prediction."""

    def __init__(
        self,
        model_config: BLINSYNModelConfig,
    ):
        super().__init__(model_config)

    def predict_impl(self, node_lists: Union[list[int], np.ndarray]):
        a_nodes, b_nodes, c_nodes = node_lists
        features = self.fe.extract_features(a_nodes, b_nodes, c_nodes)
        preds = self.model.predict_proba(features)[:, 1]
        return preds

    def destroy(self):
        del self.model
        del self.fe

    def summary(self):
        raise NotImplementedError

    def _build_model(self):
        return LogisticRegression(random_state=0, max_iter=10000)

    def _build_feature_extractor(self):
        return BLINSYNFeatureExtractor(
            self.model_config.drug_feature_name, self.model_config.cell_feature_name
        )


class BLINSYNHandlerFactory(HandlerFactory):
    def __init__(self, model_config: BLINSYNModelConfig) -> None:
        super().__init__()
        self.model_config = model_config

    def create_handler(self) -> BLINSYNModelHandler:
        return BLINSYNModelHandler(self.model_config)

# Optimisation

In [ ]:
import numpy as np

from netbalance.data.association_data import TGData
from netbalance.evaluation import Result
from netbalance.evaluation.utils import evaluate_binary_classification_simple
from netbalance.optimization.interface import Trainer


class BLINSYNTrainer(Trainer):

    def train(
        self,
        model_handler: BLINSYNModelHandler,
        data: TGData,
        config: BLINSYNOptimizerConfig,
    ) -> Result:
        model_handler.fe.build()

        associations = data.associations
        dp_embed = model_handler.fe.extract_features(
            associations[:, 0], associations[:, 1], associations[:, 2]
        )
        y = np.array(associations[:, -1].tolist(), dtype=np.float32).reshape(-1)
        model_handler.model.fit(dp_embed, y)

        preds = model_handler.predict(
            [data.associations[:, i] for i in range(data.associations.shape[1] - 1)]
        )
        result = evaluate_binary_classification_simple(
            data.associations[:, -1], preds.reshape(-1), config.threshold
        )
        return result

# Train and Evaluate

## Stage 1

In [ ]:
import os

from netbalance.data.association_data import TGData, TGTrainTestSpliter
from netbalance.evaluation import repeated_cross_validation
from netbalance.features.sanger import SangerDataset as Dataset  # Parameter

model_name = "blinsyn"  # Parameter
dataset = "sanger"  # Parameter
train_neg_samp_method = "beta"  # Parameter

splitter_kwargs = {
    "k": 5,
    "train_balance": True,  # Parameter
    "train_balance_kwargs": {
        "balance_method": train_neg_samp_method,
        "negative_ratio": 1.0,  # Parameter
    },
}

model_result_dir = os.path.join(
    BLINSYN_RESULTS_DIR,
    f"preds",
    f"dataset_{dataset}",
    f"train_neg_samp_{train_neg_samp_method}",
)

ds = Dataset()


def get_data():
    return TGData(
        associations=ds.get_associations(with_negatives=False),  # Parameters
        cluster_a_node_names=ds.get_cluster_a_node_names(),
        cluster_b_node_names=ds.get_cluster_b_node_names(),
        cluster_c_node_names=ds.get_cluster_c_node_names(),
    )


repeated_cross_validation(
    get_data=get_data,
    SplitterClass=TGTrainTestSpliter,
    handler_factory=BLINSYNHandlerFactory(model_config=BLINSYNModelConfig()),
    trainer=BLINSYNTrainer(),
    optimizer_config=BLINSYNOptimizerConfig(),
    num_cross_validation=5,
    save_preds_dir=model_result_dir,
    splitter_kwargs=splitter_kwargs,
    parallel=False,
)

## Stage 2